# Frogger DQN "overkill" (píxeles, A100)

Entrena el DQN de píxeles (CNN + Double DQN + n-step 3, replay de frames) sobre `ALE/Frogger` con un
presupuesto grande (por defecto 20 M pasos del entorno) y reward escalado (`scale` 0.1, la mejor
configuración hasta ahora). Los checkpoints, el leaderboard y las gráficas se guardan en Google Drive.

**Advertencias**

- No se garantiza una GPU A100: Colab la asigna según disponibilidad. Revise la celda de hardware antes de entrenar.
- Las unidades de cómputo se consumen mientras la sesión está activa. Un entrenamiento completo puede agotar su cuota.
- El entorno (emulador ALE) corre en CPU: medido en un Ryzen 5 3600, unos 2.191 pasos/s con un solo env. El aprendizaje corre en GPU. Por eso el número de vCPU importa.
- Los checkpoints se guardan en Drive cada 250.000 pasos: una desconexión pierde como máximo 250.000 pasos.

## 1. Hardware y configuración

In [ ]:
import os
import subprocess
import sys

import psutil
import torch

GB = 1024**3
BYTES_PER_TRANSITION = 7_100  # frame replay stores about 7.1 KB per transition

try:
    smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=False)
    print(smi.stdout or smi.stderr)
except FileNotFoundError:
    print("nvidia-smi not found: no NVIDIA GPU is visible")

cpu_count = os.cpu_count() or 1
ram_gb = psutil.virtual_memory().total / GB
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / GB
else:
    gpu_name, vram_gb = "none", 0.0

# Recommended settings derived from the hardware.
num_envs = min(32, max(8, cpu_count * 2))
num_envs = num_envs // 4 * 4  # multiple of 4
recommended = {
    "num_envs": num_envs,
    "num_threads": cpu_count,
    "buffer_size": 1_000_000 if ram_gb >= 40 else 500_000,
    "batch_size": 512 if vram_gb >= 30 else 256,
}

# ---- EDIT HERE ----------------------------------------------------------------
# Keys left out keep the recommended (or default) value.
USER_OVERRIDES = {
    # "num_envs": 16,
    # "batch_size": 256,
    # "buffer_size": 500_000,
    # "life_loss_terminal": True,  # bootstrap-only terminal on a lost life (H12)
}
# PRESET "replicate_run3": same settings as local run 3 (8 envs, batch 128, buffer 200k,
# reward scale 0.1, 5M steps) with a different seed, to test whether reaching home is robust.
# PRESET "overkill": use everything the hardware allows (A100 / H100 class runtimes).
PRESET = "replicate_run3"
PRESETS = {
    "replicate_run3": {
        "run_name": "px_r3_seed43",
        "total_steps": 5_000_000,
        "num_envs": 8,
        "num_threads": min(8, cpu_count),
        "buffer_size": 200_000,
        "batch_size": 128,
        "seed": 43,
        "life_loss_terminal": False,
    },
    "overkill": {**recommended, "run_name": "px_overkill", "total_steps": 20_000_000, "seed": 42},
}
DEFAULTS = {"reward_scale": 0.1, "save_interval": 250_000, "life_loss_terminal": False}
CONFIG = {**DEFAULTS, **PRESETS[PRESET], **USER_OVERRIDES}
# -------------------------------------------------------------------------------

# The trainer requires total_steps to be a multiple of num_envs.
CONFIG["total_steps"] = CONFIG["total_steps"] // CONFIG["num_envs"] * CONFIG["num_envs"]

replay_gb = CONFIG["buffer_size"] * BYTES_PER_TRANSITION / 1e9
print(f"GPU: {gpu_name}, VRAM {vram_gb:.1f} GB")
print(f"CPU: {cpu_count} vCPU, RAM {ram_gb:.1f} GB")
print(f"Replay estimate: {CONFIG['buffer_size']:,} transitions x 7.1 KB = {replay_gb:.2f} GB")
if vram_gb == 0:
    print("WARNING: no CUDA GPU. Use Runtime > Change runtime type > A100 GPU.")
if replay_gb + 3 > ram_gb:
    print("WARNING: replay plus the 3 GB margin exceeds RAM; the trainer will refuse. Lower buffer_size.")
print("\nChosen CONFIG:")
for key, value in CONFIG.items():
    print(f"  {key:<15} = {value}")

## 2. Google Drive

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
OUTPUT_DIR = f"/content/drive/MyDrive/frogger-dqn/{CONFIG['run_name']}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("OUTPUT_DIR =", OUTPUT_DIR)

## 3. Repositorio y dependencias

Se usa el torch CUDA que ya trae Colab. No se ejecuta `uv sync`, porque el `pyproject` fija un índice
de Windows cu128.

In [ ]:
import shutil
from importlib.metadata import version

REPO_URL = "https://github.com/AlejoMoncada/frogger-dqn-lab.git"
BRANCH = "exp/home-challenge"
REPO_DIR = "/content/frogger-dqn-lab"

shutil.rmtree(REPO_DIR, ignore_errors=True)
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "ale-py==0.12.1", "gymnasium>=1.0,<2", "numpy", "matplotlib", "imageio",
        "pillow", "psutil", "opencv-python-headless",
    ],
    check=True,
)

import ale_py  # noqa: E402
import gymnasium  # noqa: E402
import torch  # noqa: E402
from ale_py.vector_env import AtariVectorEnv  # noqa: E402

print("ale-py", version("ale-py"), "| gymnasium", gymnasium.__version__, "| torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("AtariVectorEnv import OK:", AtariVectorEnv.__name__)
assert torch.cuda.is_available(), "No CUDA GPU: select an A100 runtime and run again."


def stream(cmd, cwd=REPO_DIR):
    """Run a command, print its output live, and raise if it fails."""
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    proc = subprocess.Popen(
        cmd, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    )
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"command failed (exit {proc.returncode}): {' '.join(cmd)}")

## 4. Smoke test (about 1 minute)

Valida entorno, CUDA, aprendizaje, checkpoints y evaluación.

In [ ]:
shutil.rmtree("/content/smoke", ignore_errors=True)
stream([sys.executable, "train_pixels.py", "--smoke", "--device", "cuda", "--output-dir", "/content/smoke"])

## 5. Entrenamiento

Si ya existe `checkpoints/latest.pt` en Drive, la celda reanuda automáticamente. Tras una desconexión,
basta con volver a ejecutar todas las celdas.

In [ ]:
latest_path = os.path.join(OUTPUT_DIR, "checkpoints", "latest.pt")
resuming = os.path.exists(latest_path)
if not resuming and os.listdir(OUTPUT_DIR):
    raise RuntimeError(
        f"{OUTPUT_DIR} has files but no checkpoints/latest.pt. Change run_name or clear the folder in Drive."
    )

cmd = [
    sys.executable, "train_pixels.py",
    "--device", "cuda",
    "--output-dir", OUTPUT_DIR,
    "--total-steps", str(CONFIG["total_steps"]),
    "--num-envs", str(CONFIG["num_envs"]),
    "--num-threads", str(CONFIG["num_threads"]),
    "--buffer-size", str(CONFIG["buffer_size"]),
    "--batch-size", str(CONFIG["batch_size"]),
    "--reward-mode", "scale",
    "--reward-scale", str(CONFIG["reward_scale"]),
    "--save-interval", str(CONFIG["save_interval"]),
    "--seed", str(CONFIG["seed"]),
]
if CONFIG["life_loss_terminal"]:
    cmd.append("--life-loss-terminal")
if resuming:
    print(f"Resuming from {latest_path}")
    cmd.append("--resume")
else:
    print("Starting a new run")
stream(cmd)

## 6. Leaderboard y gráficas

In [ ]:
import glob
import re

from IPython.display import Image, Markdown, display

stream([sys.executable, "milestones.py", "--run-dir", OUTPUT_DIR, "--episodes", "20"])
stream([sys.executable, "plot.py", "--input-dir", OUTPUT_DIR])

with open(os.path.join(OUTPUT_DIR, "leaderboard.md"), encoding="utf-8") as f:
    display(Markdown(f.read()))
display(Image(os.path.join(OUTPUT_DIR, "learning_curve.png")))
display(Image(os.path.join(OUTPUT_DIR, "milestones.png")))

gifs = glob.glob(os.path.join(OUTPUT_DIR, "gifs", "step_*.gif"))
if gifs:
    last_gif = max(gifs, key=lambda p: int(re.search(r"step_(\d+)\.gif$", p).group(1)))
    print("Last checkpoint GIF:", last_gif)
    display(Image(last_gif))

## 7. Resultados

Todo queda en Google Drive, en la carpeta `frogger-dqn/<RUN_NAME>`: checkpoints, CSV, `leaderboard.md`,
gráficas y GIFs. No hace falta descargar nada desde la sesión; los archivos sobreviven a la desconexión.